# Prompt Caching and Context Caching Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Anthropic prompt caching with explicit markers

In [ ]:
```python

import anthropic

client = anthropic.Anthropic()

SYSTEM = [

    {

        "type": "text",

        "text": "You are a senior Python reviewer. Follow the rubric exactly.\n\n" + RUBRIC_15K_TOKENS,

        "cache_control": {"type": "ephemeral"},

    }

]

def review(code: str):

    return client.messages.create(

        model="claude-opus-4-7",

        max_tokens=1024,

        system=SYSTEM,

        messages=[{"role": "user", "content": code}],

    )

In [ ]:
```

The `cache_control` marker tells Anthropic to store the block for 5 minutes. Reuse within that window hits; reuse after expires and writes again.

**Response usage fields:**

In [ ]:
```python

response = review(code_a)

response.usage

# InputTokensUsage(

#     input_tokens=120,

#     cache_creation_input_tokens=15023,   # paid at 1.25x

#     cache_read_input_tokens=0,

#     output_tokens=340,

# )

response_b = review(code_b)

response_b.usage

# cache_creation_input_tokens=0

# cache_read_input_tokens=15023           # paid at 0.1x

In [ ]:
```

Check both fields in CI — if `cache_read_input_tokens` stays at zero across requests, your cache keys are drifting.

### Step 2: one-hour extended TTL

For long-running batch jobs, the 5-minute default expires between jobs. Set `ttl`:

In [ ]:
```python

{"type": "text", "text": RUBRIC, "cache_control": {"type": "ephemeral", "ttl": "1h"}}

In [ ]:
```

1-hour TTL costs 2x the write premium (50% over baseline instead of 25%) but pays back fast on any batch reusing the prefix more than 5 times.

### Step 3: OpenAI automatic caching

OpenAI gives you nothing to configure. Any prefix over 1,024 tokens that matches a recent request gets a 50% discount automatically.

In [ ]:
```python

from openai import OpenAI

client = OpenAI()

resp = client.chat.completions.create(

    model="gpt-5",

    messages=[

        {"role": "system", "content": SYSTEM_PROMPT},   # long and stable

        {"role": "user", "content": user_msg},

    ],

)

resp.usage.prompt_tokens_details.cached_tokens  # the discounted portion

In [ ]:
```

Same cache-friendly layout rule applies. Two things kill OpenAI's cache that don't kill Anthropic's: changing the `user` field (used as a cache key component) and reordering tools.

### Step 4: Gemini explicit context caching

Gemini treats the cache as a first-class object you create and name:

In [ ]:
```python

from google import genai

from google.genai import types

client = genai.Client()

cache = client.caches.create(

    model="gemini-3-pro",

    config=types.CreateCachedContentConfig(

        display_name="rubric-v3",

        system_instruction=RUBRIC,

        contents=[FEW_SHOT_EXAMPLES],

        ttl="3600s",

    ),

)

resp = client.models.generate_content(

    model="gemini-3-pro",

    contents=["Review this code:\n" + code],

    config=types.GenerateContentConfig(cached_content=cache.name),

)

In [ ]:
```

Gemini charges storage per token·hour for as long as the cache lives, and reads at ~25% of normal input rate. This is the right shape when you reuse the same giant prompt across many sessions over days.

### Step 5: measuring hit rate in production

See `code/main.py` for a simulated three-provider accountant that tracks write/read/miss counts and computes blended cost per 1K requests. Gate deploys on a target hit rate — most production Anthropic setups should see >80% read fraction after warmup.

## Exercises

In [ ]:
1. **Easy.** Take a 10-turn conversation with a 5,000-token system prompt against Claude. Run it without `cache_control` and then with. Report the input-token bill for each.
2. **Medium.** Write a test harness that, given a prompt template and a request log, computes the expected hit rate and dollar savings per provider (Anthropic 5m, Anthropic 1h, OpenAI automatic, Gemini explicit).
3. **Hard.** Build a layout optimizer: given a prompt and a list of fields marked `stable=True/False`, rewrite the prompt to put a single cache breakpoint at the maximum cache-friendly position without losing information. Verify on a real Anthropic endpoint.